# PaperLens: Exploratory Data Analysis & NLP Pipeline
**Course:** BCA 5th Semester NLP Project  
**Topic:** Scientific Paper Summarization, Keyword Extraction & ROUGE Evaluation

This notebook demonstrates exploratory data analysis on the arXiv scientific paper dataset, breaks down each stage of our text preprocessing pipeline, and compares three extractive summarization techniques (**Lead Baseline**, **TF-IDF**, and **TextRank**) using standard **ROUGE** evaluation metrics.

In [ ]:
import os
import sys
import pandas as pd
import matplotlib.pyplot as plt
import nltk

# Ensure project root is in sys.path
PROJECT_ROOT = os.path.abspath("..")
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

# Ensure required NLTK tokenizers and stopwords are available
for pkg in ["punkt", "punkt_tab", "stopwords", "wordnet"]:
    nltk.download(pkg, quiet=True)

# Load sample dataset
dataset_path = os.path.join(PROJECT_ROOT, "dataset", "dataset.csv")
df = pd.read_csv(dataset_path)
print(f"Loaded {len(df)} papers from {dataset_path}")
df.head(2)

## 1. Document & Abstract Length Analysis
Scientific research papers contain several thousand words, whereas abstracts compress this information into roughly 100-300 words. Let us analyze this compression ratio.

In [ ]:
df["article_words"] = df["article"].apply(lambda x: len(str(x).split()))
df["abstract_words"] = df["abstract"].apply(lambda x: len(str(x).split()))
df["compression_pct"] = (df["abstract_words"] / df["article_words"]) * 100

print("Summary Statistics:")
df[["article_words", "abstract_words", "compression_pct"]].describe()

In [ ]:
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.hist(df["article_words"], bins=15, color="#4C72B0", edgecolor="black")
plt.title("Article Word Count Distribution")
plt.xlabel("Words")
plt.ylabel("Number of Papers")

plt.subplot(1, 2, 2)
plt.hist(df["abstract_words"], bins=15, color="#55A868", edgecolor="black")
plt.title("Abstract Word Count Distribution")
plt.xlabel("Words")
plt.tight_layout()
plt.show()

## 2. Text Preprocessing Walkthrough
Raw text requires cleaning (cutting citations, placeholders, and lone page numbers), sentence splitting, and normalization (lemmatization vs stemming).

In [ ]:
from src.preprocessing import clean_text, split_sentences, preprocess_sentence

sample_raw = df.loc[0, "article"]
sample_clean = clean_text(sample_raw)
sentences = split_sentences(sample_clean)

print(f"Raw characters:      {len(sample_raw):,}")
print(f"Cleaned characters:  {len(sample_clean):,}")
print(f"Valid sentences:     {len(sentences)}")
print("\nFirst sentence:", sentences[0])

In [ ]:
# Demonstrating Lemmatization vs Porter Stemming
lemma_tokens = preprocess_sentence(sentences[0], method="lemma")
stem_tokens = preprocess_sentence(sentences[0], method="stem")

print(f"{'Original Token':<20} | {'Lemmatized':<15} | {'Stemmed'}")
print("-" * 55)
for orig, lem, stm in zip(sentences[0].split()[:8], lemma_tokens[:8], stem_tokens[:8]):
    print(f"{orig:<20} | {lem:<15} | {stm}")

## 3. Comparing Summarization Methods
We evaluate three extractive approaches:
1. **Lead Baseline**: Heuristic selection of opening sentences.
2. **TF-IDF**: Sentence scoring based on mean word saliency.
3. **TextRank**: Graph centrality (PageRank on sentence similarity network).

In [ ]:
from src.model import summarize

summaries = summarize(sentences, mode="compare", length="medium")
for method, text in summaries.items():
    print(f"=== {method.upper()} ({len(text.split())} words) ===")
    print(text[:280] + "...\n")

## 4. ROUGE Evaluation Against Author's Abstract
We score the generated summaries against the actual author abstract using ROUGE-1 (unigram overlap), ROUGE-2 (bigram overlap), and ROUGE-L (Longest Common Subsequence).

In [ ]:
from src.evaluation import calculate_rouge

reference = df.loc[0, "abstract"]
print("Author Abstract Reference:")
print(reference[:250] + "...\n")

print("ROUGE F1 Scores:")
for method, text in summaries.items():
    scores = calculate_rouge(reference, text)
    print(f"{method.upper():<10}: {scores}")